# OSA Alerts Pipeline — Full Run

Runs the complete pipeline end-to-end in a single notebook.  
Each stage is a separate cell so you can inspect intermediate outputs before proceeding.

| Cell | Stage | What happens |
|------|-------|--------------|
| 0 | Config | Set analysis date, optional store filter |
| 1 | Session | Connect to Snowflake |
| 2 | Data load | OSA + hierarchy joined, cached |
| 3 | Stage 1 — Detect | Anomaly signals (WOW / VS_CHAIN / REPEAT) |
| 4 | Stage 2 — Facts | Enrich signals with PPG, DoW, chain drift, repeat history |
| 5 | Stage 3 — Investigate | Generate narratives (rule-based by default) |
| 6 | Stage 4 — Score | Assign priority scores |
| 7 | Preview | Read through top alert cards |
| 8 | Write | Persist to Snowflake — **run manually when satisfied** |

In [ ]:
# ── Date window ────────────────────────────────────────────────────────────────
# Set ANALYSIS_DATE_STR to the last day of the week you want to analyse.
# All other windows are derived automatically.
ANALYSIS_DATE_STR = '2025-06-14'

# ── Client config (None = base defaults; 'eastofengland' = client override) ────
CLIENT = None   # or 'eastofengland'

# ── Optional: restrict to one store for a faster test run ─────────────────────
QC_STORE_ID = None   # e.g. 507 — set to None for all stores

# ── Threshold overrides (None = use config; values override config) ──────────
MIN_WOW_DROP_OVERRIDE           = None   # e.g. 0.01
MIN_CHAIN_GAP_OVERRIDE          = None   # e.g. 0.03
MIN_LOST_SALES_VALUE_OVERRIDE   = None   # e.g. 200
MIN_REPEAT_ALERT_WEEKS_OVERRIDE = None   # e.g. 4
PROJECTION_WEEKS       = 4      # cost-if-not-fixed projection horizon
DRILLDOWN_TOP_N          = 3      # top-N PPGs in drilldown lists

import datetime
analysis_date   = datetime.date.fromisoformat(ANALYSIS_DATE_STR)
THIS_WEEK_END   = str(analysis_date)
THIS_WEEK_START = str(analysis_date - datetime.timedelta(days=6))
LAST_WEEK_END   = str(analysis_date - datetime.timedelta(days=7))
LAST_WEEK_START = str(analysis_date - datetime.timedelta(days=13))
LOOKBACK_START  = str(analysis_date - datetime.timedelta(days=69))  # ~10 weeks
RUN_DATE        = str(datetime.date.today())

print(f'Analysis date : {THIS_WEEK_END}')
print(f'Run date      : {RUN_DATE}')
print(f'This week     : {THIS_WEEK_START} – {THIS_WEEK_END}')
print(f'Last week     : {LAST_WEEK_START} – {LAST_WEEK_END}')
print(f'Lookback      : {LOOKBACK_START} – {THIS_WEEK_END}')
print(f'Client        : {CLIENT or "base defaults"}')
print(f'Store filter  : {QC_STORE_ID if QC_STORE_ID else "all stores"}')

Analysis date : 2025-06-14
Run date      : 2026-04-01
This week     : 2025-06-08 – 2025-06-14
Last week     : 2025-06-01 – 2025-06-07
Lookback      : 2025-04-06 – 2025-06-14
Store filter  : all stores


In [ ]:
import sys, os, logging, warnings

_root = os.path.abspath(os.path.join('..', '..', '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

# ── Suppress noisy Snowflake connector output ─────────────────────────────────
logging.getLogger('snowflake.connector').setLevel(logging.ERROR)
logging.getLogger('snowflake.snowpark').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='snowflake')

from store_insight.alerts.config.loader import load_config
from store_insight.alerts.pipeline.session import create_session

cfg     = load_config(CLIENT)
print(f'Config loaded: client={cfg["client_name"]}')

session = create_session(cfg)   # prints 'Snowflake session created.'

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/93a31d5d-1a5d-433f-baa0-138549cd2108/saml2?SAMLRequest=nZLBctowEIZfxaOebUm2SUEDZCg0LW2gTDBN2ptiy0QgS0Yrx0mfvsKEmfSQHHrxeORv15%2F23%2BHlU6WCR2FBGj1CNCIoEDo3hdTbEdpkV2EfBeC4LrgyWozQswB0OR4Cr1TNJo170Dfi0AhwgW%2BkgXUfRqixmhkOEpjmlQDmcraeLK5ZHBFWW%2BNMbhR6VfJ%2BBQcQ1nnDc0kB0us9OFczjNu2jdokMnaLY0IIJgPsqSPy4cw%2F%2BTu9wVNM0iPvCY%2BvXtw%2BSX0awXta9ycI2NcsW4WrH%2BsMBZOz6tRoaCph18I%2Bylxsbq5PAuANoKIXCUmjZh%2BC8TOM%2BJ%2FGigi0aUvF9yI3Vd043zryb7gUBVZmK%2F3t57MRqveyuL2Vvw53%2FWm9W5f0e18d5CG%2FW3wp5efsW7XfbTbLdJ8vJ7tkATkKfp7jjY%2FxzgEaMdfHUJ0%2FIvFFSNKQ0IwSllJGP0a0R3%2BjYOZDlZq7rvJs3nlElcytAVM6o5XUorMcJDyhRa8IKfePNEnK8J5zEtKk30sHeRFT0sfHqGN0Wh%2FWidjxfw1liF%2B3eFnHpU9oPlsZJfPn4MrYiru3A6QR7U5kEZYdykTFpZoUhRUAPkilTDu1gju%2F

 pip install snowflake-connector-python[secure-local-storage]


In [ ]:
from snowflake.snowpark import functions as F

# ── Resolve table names from config ──────────────────────────────────────────
OSA_TABLE       = cfg['tables']['osa']
HIERARCHY_TABLE = cfg['tables']['hierarchy']
FEEDBACK_TABLE  = cfg['tables']['feedback']
OUTPUT_TABLE    = cfg['output']['alerts']

print(f'OSA table      : {OSA_TABLE}')
print(f'Hierarchy table: {HIERARCHY_TABLE}')
print(f'Feedback table : {FEEDBACK_TABLE}')
print(f'Output table   : {OUTPUT_TABLE}')

osa_raw = (
    session.table(OSA_TABLE)
    .filter(F.col('ANALYSIS_DATE') >= LOOKBACK_START)
    .filter(F.col('ANALYSIS_DATE') <= THIS_WEEK_END)
    .select(
        F.col('SITE_ID').alias('STORE_ID'),
        F.col('ITEM_ID').alias('ITEM_CODE'),
        F.col('ACTUAL_SALES_VOLUME'),
        F.col('LOST_SALES_VOLUME'),
        F.col('LOST_SALES_VALUE'),
        F.col('ANALYSIS_DATE'),
        F.col('DEPARTMENT_ID'),
    )
)

item = (
    session.table(HIERARCHY_TABLE)
    .select(
        F.col('MASTER_ITEM_ID').alias('ITEM_CODE'),
        F.col('ITEM_ID'),
        F.col('DEPARTMENT_NAME'),
        F.col('SUBCLASS_NAME'),
        F.col('PRODUCT_PROFILE_GROUP_NAME'),
    )
    .distinct()
)

osa = (
    osa_raw
    .join(item, on=['ITEM_CODE'], how='left')
    .filter(F.col('DEPARTMENT_NAME').is_not_null())
    .cache_result()
)

feedback = (
    session.table(FEEDBACK_TABLE)
    .filter(F.col('ALERT_ANALYSIS_DATE') >= LAST_WEEK_START)
    .filter(F.col('ALERT_ANALYSIS_DATE') <= THIS_WEEK_END)
    .select(
        F.col('SITE_ID').alias('STORE_ID'),
        F.col('SITE_NAME'),
        F.col('ALERT_TYPE_NAME'),
        F.col('ALERT_ANALYSIS_DATE'),
        F.col('INTERVENTION_ID'),
        F.col('TICK_AND_FLICK_FLAG'),
    )
    .cache_result()
)

print(f'\nOSA rows loaded & cached      : {osa.count():,}')
print(f'Feedback rows loaded & cached : {feedback.count():,}')

OSA rows loaded & cached      : 73,588,943
Feedback rows loaded & cached : 22,275


## Stage 1 — Anomaly Detection
Runs WOW, VS_CHAIN, and REPEAT detectors then merges results.

In [ ]:
from store_insight.alerts.stages._1_detect.detect import detect_signals
from store_insight.alerts.pipeline.config import DetectorConfig
from collections import Counter

data = osa.filter(F.col('STORE_ID') == QC_STORE_ID) if QC_STORE_ID else osa

# ── Build DetectorConfig from config with optional overrides ────────────────
cfg_thresholds = cfg.get('thresholds', {})
cfg_min_wow_drop           = MIN_WOW_DROP_OVERRIDE or cfg_thresholds.get('min_wow_drop', 0.01)
cfg_min_chain_gap          = MIN_CHAIN_GAP_OVERRIDE or cfg_thresholds.get('min_chain_gap', 0.03)
cfg_min_lost_sales_value   = MIN_LOST_SALES_VALUE_OVERRIDE or cfg_thresholds.get('min_lost_sales_value', 200.0)
cfg_min_alert_weeks        = MIN_REPEAT_ALERT_WEEKS_OVERRIDE or cfg_thresholds.get('min_alert_weeks', 4)

det_cfg = DetectorConfig(
    min_lost_sales_value=cfg_min_lost_sales_value,
    min_wow_drop=cfg_min_wow_drop,
    min_chain_gap=cfg_min_chain_gap,
    min_alert_weeks=cfg_min_alert_weeks,
)

print(f'DetectorConfig: min_wow_drop={det_cfg.min_wow_drop}, min_chain_gap={det_cfg.min_chain_gap}, '
      f'min_lost_sales_value={det_cfg.min_lost_sales_value}, min_alert_weeks={det_cfg.min_alert_weeks}')

signals = detect_signals(
    data,
    last_week_start=LAST_WEEK_START,
    last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START,
    this_week_end=THIS_WEEK_END,
    lookback_start=LOOKBACK_START,
    feedback=feedback,
    config=det_cfg,
    cfg=cfg,
)

trigger_counts = Counter(tuple(sorted(s['trigger_types'])) for s in signals)
print(f'\nTotal signals: {len(signals)}')
for triggers, count in sorted(trigger_counts.items()):
    print(f'  {"+".join(triggers):<30} {count}')

print('\nTop 5 signals by LSV:')
for s in sorted(signals, key=lambda x: x.get('this_week_lsv', 0), reverse=True)[:5]:
    print(f"  store={s['store_id']:>4}  dept={s.get('department', s.get('signal_type', '')):<30}  "
          f"triggers={s['trigger_types']}  lsv=£{s.get('this_week_lsv', 0):,.2f}")

[1_detect] TICK_AND_FLICK=13 (chain avg 35.4%)
[1_detect] 587 signals — REPEAT=321, VS_CHAIN=252, WOW=224, TICK_AND_FLICK=13
Total signals: 587
  REPEAT                         228
  REPEAT+VS_CHAIN                31
  REPEAT+VS_CHAIN+WOW            37
  REPEAT+WOW                     25
  TICK_AND_FLICK                 13
  VS_CHAIN                       91
  VS_CHAIN+WOW                   93
  WOW                            69

Top 5 signals by LSV:
  store= 239  dept=CIGARETTES                      triggers=['REPEAT', 'VS_CHAIN', 'WOW']  lsv=£3,590.43
  store= 214  dept=CIGARETTES                      triggers=['REPEAT', 'VS_CHAIN']  lsv=£2,977.52
  store= 430  dept=CIGARETTES                      triggers=['REPEAT']  lsv=£2,218.03
  store= 207  dept=CIGARETTES                      triggers=['REPEAT']  lsv=£2,074.31
  store= 508  dept=SOFT DRINKS                     triggers=['REPEAT', 'VS_CHAIN', 'WOW']  lsv=£1,933.14


## Stage 2 — Fact Gathering
Enriches each signal with PPG drilldowns, day-of-week pattern, chain drift, and repeat item history.

In [ ]:
from store_insight.alerts.stages._2_facts.facts import enrich_signals

enriched = enrich_signals(
    signals,
    osa,   # always pass full chain osa for chain-level gatherers
    last_week_start=LAST_WEEK_START,
    last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START,
    this_week_end=THIS_WEEK_END,
    lookback_start=LOOKBACK_START,
    projection_weeks=PROJECTION_WEEKS,
    drilldown_top_n=DRILLDOWN_TOP_N,
    cfg=cfg,
)

# QC checks
missing_ppg   = sum(1 for s in enriched if not s['facts']['subclass_contributors'])
missing_dow   = sum(1 for s in enriched if not s['facts']['dow_pattern'])
missing_chain = sum(1 for s in enriched if s['facts']['chain_this_week_osa'] is None)

print(f'{len(enriched)} signals enriched')
print(f'  No PPG contributors : {missing_ppg}')
print(f'  No DoW pattern      : {missing_dow}')
print(f'  No chain OSA        : {missing_chain}')

# Spot check — top signal
top = max(enriched, key=lambda x: x['this_week_lsv'])
f   = top['facts']
print(f"\nSpot check — store={top['store_id']}  dept={top['department']}")
print(f"  worst_day        : {f['worst_day']}")
print(f"  chain_this_wk_osa: {f['chain_this_week_osa']}%")
print(f"  cost_projection  : £{f['cost_projection']:,.2f}")
if f['subclass_contributors']:
    print(f"  top PPG          : {f['subclass_contributors'][0]['subclass_name']} "
          f"£{f['subclass_contributors'][0]['this_week_lsv']:,.2f}")

## Stage 3 — Narrative Generation
Uses rule-based templates by default.  
To use Cortex instead, set `_USE_CORTEX = True` in `_3_investigate/investigate.py`.

In [ ]:
from store_insight.alerts.stages._3_investigate.investigate import investigate

narratives = [investigate(sig, session=session, cfg=cfg) for sig in enriched]

required = ['issue', 'explanation', 'cost_if_not_fixed', 'recommended_action']
blank_fields = [(enriched[i]['store_id'], enriched[i]['department'], k)
                for i, n in enumerate(narratives)
                for k in required if not n.get(k)]

print(f'{len(narratives)} narratives generated')
if blank_fields:
    print(f'WARNING — {len(blank_fields)} blank fields:')
    for b in blank_fields[:5]:
        print(f'  store={b[0]}  dept={b[1]}  missing={b[2]}')
else:
    print('All four fields populated for every narrative.')

## Stage 4 — Scoring
Assigns `sort_value = this_week_lsv × trigger_count_weight` (1/2/3 triggers → weight 1/2/3).

In [ ]:
from store_insight.alerts.stages._4_score.score import score
import statistics

cards = []
for sig, nar in zip(enriched, narratives):
    sort_value  = score({**sig, **nar})
    trigger_str = ','.join(sorted(sig['trigger_types']))
    cards.append({
        'alert_type':         f'OSA_{trigger_str}',
        'store_id':           sig['store_id'],
        'section':            sig['department'],
        'type':               'Availability',
        'trigger_types':      sig['trigger_types'],
        'issue':              nar['issue'],
        'explanation':        nar['explanation'],
        'cost_if_not_fixed':  nar['cost_if_not_fixed'],
        'recommended_action': nar['recommended_action'],
        'sort_value':         sort_value,
    })

cards.sort(key=lambda x: x['sort_value'], reverse=True)
scores = [c['sort_value'] for c in cards]

print(f'{len(cards)} alert cards scored and sorted')
print(f'Score range: £{min(scores):,.2f} – £{max(scores):,.2f}')
print(f'Median score: £{statistics.median(scores):,.2f}')

## Preview — Top Alert Cards
Read through the highest-priority alerts before writing to Snowflake.

In [ ]:
PREVIEW_N = 10

print(f"{'Rank':>4}  {'Store':>5}  {'Department':<30}  {'Triggers':<25}  {'Score':>10}")
print(f"{'----':>4}  {'-----':>5}  {'-'*30}  {'-'*25}  {'-'*10}")
for i, c in enumerate(cards[:PREVIEW_N], 1):
    triggers = '+'.join(c['trigger_types'])
    print(f"{i:>4}  {c['store_id']:>5}  {c['section']:<30}  {triggers:<25}  {c['sort_value']:>10,.2f}")

In [ ]:
# ── Print full narrative for any card by rank (1-based) ───────────────────────
RANK = 1

c = cards[RANK - 1]
print('=' * 65)
print(f"RANK {RANK}  |  Store {c['store_id']}  |  {c['section']}  |  {'+'.join(c['trigger_types'])}")
print(f"Score: £{c['sort_value']:,.2f}")
print('=' * 65)
print(f"\nISSUE\n{c['issue']}")
print(f"\nEXPLANATION\n{c['explanation']}")
print(f"\nCOST IF NOT FIXED\n{c['cost_if_not_fixed']}")
print(f"\nRECOMMENDED ACTION\n{c['recommended_action']}")

## Write to Snowflake

> **Only run this cell when you are satisfied with the output above.**  
> Set `OVERWRITE = True` to replace all rows for this `RUN_DATE` (safe to re-run).  
> Set `OVERWRITE = False` to append (use when running multiple store batches).
>
> Output table is resolved from config (`cfg['output']['alerts']`).

In [ ]:
OVERWRITE = False   # set True to overwrite existing rows for this run date

from store_insight.alerts.pipeline.pipeline import write_alerts
write_alerts(session, cards, run_date=RUN_DATE, overwrite=OVERWRITE, cfg=cfg)